# ProveTok Stage0-4 Output Analysis (Colab)

这个 notebook 用于分析 `run_mini_experiment.py` 产出的结果（450/450 或 smoke）。

输入目录需要包含：
- `summary.csv`
- `ctrate_case_summary.csv`
- `radgenome_case_summary.csv`
- `run_meta.json`
- `cases/*/*/trace.jsonl`


In [ ]:
# Colab dependencies
!pip -q install pandas numpy matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 修改成你的输出目录
OUT_DIR = Path('/content/drive/MyDrive/Data/outputs_stage0_4_full450_cp_strict')
assert OUT_DIR.exists(), f'Not found: {OUT_DIR}'
print('OUT_DIR =', OUT_DIR)

## 1) 先做结构验收

In [ ]:
# 如果仓库不在当前会话，先 clone 一份
%cd /content
!test -d ProveTok_ACM || git clone https://github.com/Ricardo-ChenTY/ProveTok_ACM.git
%cd /content/ProveTok_ACM

!python validate_stage0_4_outputs.py \
  --out_dir "$OUT_DIR" \
  --datasets ctrate,radgenome \
  --expected_cases_map ctrate=450,radgenome=450 \
  --save_report "$OUT_DIR/validation_report.json"

In [ ]:
run_meta_path = OUT_DIR / 'run_meta.json'
assert run_meta_path.exists(), 'run_meta.json missing'
run_meta = json.loads(run_meta_path.read_text(encoding='utf-8'))

print('cp_strict =', run_meta.get('cp_strict'))
print('text_encoder =', run_meta.get('text_encoder'))
print('ctrate selected/processed =', run_meta['ctrate']['selected_rows'], run_meta['ctrate']['processed_rows'])
print('radgenome selected/processed =', run_meta['radgenome']['selected_rows'], run_meta['radgenome']['processed_rows'])

## 2) 全局汇总统计

In [ ]:
summary = pd.read_csv(OUT_DIR / 'summary.csv')
display(summary.head())

agg = summary.groupby('dataset', as_index=False).agg(
    cases=('case_id', 'count'),
    avg_tokens=('n_tokens', 'mean'),
    avg_sentences=('n_sentences', 'mean'),
    avg_violations=('n_violations', 'mean'),
    total_violations=('n_violations', 'sum'),
)
display(agg)

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(agg['dataset'], agg['avg_violations'])
plt.title('Avg Violations per Case')
plt.ylabel('avg n_violations')
plt.show()

## 3) 解析 trace.jsonl（规则/句子级）

In [ ]:
from collections import Counter

trace_files = sorted((OUT_DIR / 'cases').glob('*/*/trace.jsonl'))
print('trace files:', len(trace_files))

rows = []
rule_counter = Counter()

for tf in trace_files:
    dataset = tf.parts[-3]
    case_id = tf.parts[-2]
    with tf.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if 'sentence_text' not in obj:
                continue
            vios = obj.get('violations', []) or []
            rule_ids = [v.get('rule_id', 'UNKNOWN') for v in vios if isinstance(v, dict)]
            for rid in rule_ids:
                rule_counter[rid] += 1
            rows.append({
                'dataset': dataset,
                'case_id': case_id,
                'sentence_text': obj.get('sentence_text', ''),
                'n_topk': len(obj.get('topk_token_ids', []) or []),
                'n_violations': len(vios),
                'has_violation': int(len(vios) > 0),
                'rule_ids': '|'.join(rule_ids),
            })

sent_df = pd.DataFrame(rows)
display(sent_df.head())
print('sentence rows:', len(sent_df))

In [ ]:
rule_df = pd.DataFrame([
    {'rule_id': k, 'count': v} for k, v in sorted(rule_counter.items(), key=lambda x: -x[1])
])
display(rule_df)

if not rule_df.empty:
    plt.figure(figsize=(8, 4))
    plt.bar(rule_df['rule_id'], rule_df['count'])
    plt.title('Violation Count by Rule')
    plt.xticks(rotation=30)
    plt.show()

In [ ]:
sent_agg = sent_df.groupby('dataset', as_index=False).agg(
    sentences=('sentence_text', 'count'),
    violation_sentences=('has_violation', 'sum'),
)
sent_agg['violation_sentence_rate'] = sent_agg['violation_sentences'] / sent_agg['sentences'].clip(lower=1)
display(sent_agg)

In [ ]:
case_abn = sent_df.groupby(['dataset', 'case_id'], as_index=False).agg(
    n_sentences=('sentence_text', 'count'),
    n_violation_sentences=('has_violation', 'sum'),
)
case_abn['violation_ratio'] = case_abn['n_violation_sentences'] / case_abn['n_sentences'].clip(lower=1)
case_abn = case_abn.sort_values(['violation_ratio', 'n_violation_sentences'], ascending=[False, False])
display(case_abn.head(30))

## 4) 导出分析结果（给组会/写作）

In [ ]:
analysis_dir = OUT_DIR / 'analysis_exports'
analysis_dir.mkdir(parents=True, exist_ok=True)

agg.to_csv(analysis_dir / 'dataset_aggregate.csv', index=False)
sent_agg.to_csv(analysis_dir / 'sentence_violation_rate.csv', index=False)
rule_df.to_csv(analysis_dir / 'rule_violation_count.csv', index=False)
case_abn.to_csv(analysis_dir / 'abnormal_cases_ranked.csv', index=False)

print('saved to:', analysis_dir)